# 00.01 · What an AI engineer actually does in 2026

**Module 00 · Foundations, Tooling & Research Skills** · Part 1 (Core)

| | |
|---|---|
| ⏱️ Estimated time | 45–60 min |
| 📋 Prerequisites | None. This is the first lesson. |
| 🧠 Kernel | Any Python 3 kernel (only the standard library is used) |
| ✅ Docs verified | 2026-10-11 |

> **How to use this notebook:** read each Markdown cell, then run the code cell below it with **Shift + Enter**.
> Run cells **from top to bottom**. If anything looks strange, use **Kernel → Restart Kernel and Run All Cells…**

## Learning objectives

By the end of this lesson you will be able to:

1. **Explain** the difference between *calling a model API* and *engineering a reliable LLM system*.
2. **Describe** the agent development lifecycle: **build → test → deploy → monitor**, and back again.
3. **Place** LangChain, LangGraph, Deep Agents, and LangSmith on that lifecycle, using the official "framework vs runtime vs harness" vocabulary.
4. **Define** the key terms of the course (agent, harness, context engineering, evaluation-driven development, observability) and look them up in an interactive glossary.
5. **Edit and render** Markdown cells, the first Jupyter skill of the course.

## 📚 Official reading (verified 2026-10-11)

Read these after the lesson (20–30 min total). They are the source of every claim below.

| Page | Why read it |
|---|---|
| [Runtimes, frameworks, and harnesses](https://docs.langchain.com/oss/python/concepts/products) | The official vocabulary for LangChain vs LangGraph vs Deep Agents |
| [LangChain overview](https://docs.langchain.com/oss/python/langchain/overview) | "Agent = Model + Harness", and `create_agent` |
| [LangGraph overview](https://docs.langchain.com/oss/python/langgraph/overview) | The low-level runtime: durable execution, streaming, human-in-the-loop |
| [LangChain philosophy](https://docs.langchain.com/oss/python/langchain/philosophy) | Why LangChain exists, and how it changed from 2022 to today |
| [LangSmith Observability](https://docs.langchain.com/langsmith/observability) | Traces, monitoring, and evaluation datasets |

## Setup

This lesson needs **no packages and no API keys**. Run the cell below anyway: it confirms that your notebook has a working kernel.

**How to run a cell:** click inside it, then press **Shift + Enter** (or click the ▶ *Run* button in the toolbar).

In [ ]:
# A "smoke test": if this prints, your kernel is alive and running Python.
import platform
import sys

print("Python version :", platform.python_version())
print("Running from   :", sys.executable)

**You should now see** two lines: a Python version (for example `3.13.5`) and the path of the Python program that runs this notebook.
A number such as `[1]` appears to the left of the cell. That is the **execution count**: the order in which you ran cells. You will learn why it matters in Lesson 00.03.

> **If nothing happens** and the left side shows `[*]` for a long time, the kernel is busy or not started. Use **Kernel → Restart Kernel…** and run the cell again. If the top-right corner says *No Kernel*, click it and pick a Python 3 kernel.

---
## 1. Calling an API is not the job

Calling a model takes three lines of code. You will do it in Lesson 00.07. So why do companies hire **AI engineers**?

Because the demo that worked on Friday breaks on Monday. Here are real failure modes, each covered later in the course:

| What breaks | Example | Where the course fixes it |
|---|---|---|
| The model ID disappears | Groq shut down `llama-3.3-70b-versatile` for free and developer accounts on **2026-08-16**. Tutorials still use it ([Groq deprecations](https://console.groq.com/docs/deprecations)) | Lesson 00.05, `mastery.models` (Module 01) |
| Rate limits | Groq's free plan allows **8K tokens per minute** for `openai/gpt-oss-20b` ([rate limits](https://console.groq.com/docs/rate-limits)). One long document can use that up | Lesson 00.06, Module 13 |
| Wrong or made-up answers | The model invents a citation | RAG with citations (Module 06), evaluation (Module 12) |
| Unsafe behavior | A web page tells your agent to "ignore previous instructions" (prompt injection) | Guardrails (Module 04) |
| Nobody knows why it failed | A user reports a bad answer but you have no record of what the agent did | Tracing with LangSmith (Lesson 01.08, Module 12) |
| Cost surprises | A loop calls the model 200 times | Middleware limits (Module 04), cost tracking (Module 13) |

**An AI engineer builds the system around the model** so that it is reliable, measurable, safe, affordable, and deployable.

### What the job looks like day to day

- **Orchestration:** deciding which steps a model takes, in what order, with which tools.
- **Context engineering:** choosing exactly what goes into the model's context window: instructions, retrieved documents, memory, tool results. (The model only knows what you give it.)
- **Retrieval (RAG):** connecting models to your own data, with citations.
- **Tools and agents:** letting the model call functions, APIs, and other agents, safely.
- **Evaluation:** proving with data that a change made things better, not worse.
- **Observability:** recording what happened in every run (traces) so you can debug and improve.
- **Deployment and operations:** APIs, containers, rate limits, costs, monitoring, model upgrades.
- **Research:** reading official docs, changelogs, and model cards, because this field changes every month (Lesson 00.05).

---
## 2. Agent = model + harness

The LangChain docs put it in one line: **"Agent = Model + Harness."** The *harness* is everything around the model loop: the prompt, the tools, and any middleware that shapes behavior ([LangChain overview](https://docs.langchain.com/oss/python/langchain/overview)).

The official docs also sort tools into three layers ([Runtimes, frameworks, and harnesses](https://docs.langchain.com/oss/python/concepts/products)):

| Layer | LangChain product | What it adds | Use it when |
|---|---|---|---|
| **Runtime** | **LangGraph** | Durable execution, streaming, human-in-the-loop, persistence | You need low-level control, or long-running, stateful workflows |
| **Framework** | **LangChain** (`create_agent`) | Abstractions and integrations: models, messages, tools, middleware | You want to start quickly and standardize how a team builds |
| **Harness** | **Deep Agents** | Predefined tools, prompts, planning, a filesystem, subagents | You build more autonomous agents for complex, multi-step, long-running tasks |

Two facts to remember:

1. **Both LangChain agents and Deep Agents are built on LangGraph** ([philosophy](https://docs.langchain.com/oss/python/langchain/philosophy)). Learning LangGraph early (Module 07) makes everything else less magical.
2. **LangSmith is not a library you build agents with.** It is the platform for tracing, evaluation, and deployment, and it works with any of the three ([LangSmith Observability](https://docs.langchain.com/langsmith/observability)).

---
## 3. The agent development lifecycle

The LangChain docs index ([llms.txt](https://docs.langchain.com/llms.txt)) groups its pages under **Agent development lifecycle → Build, Deploy, Monitor**. This course adds an explicit **Test** step, because evaluation is where most real projects fail:

```text
      ┌────────────┐     ┌────────────┐     ┌────────────┐     ┌────────────┐
      │   BUILD    │ ──▶ │    TEST    │ ──▶ │   DEPLOY   │ ──▶ │  MONITOR   │
      │ LangChain  │     │ LangSmith  │     │ FastAPI +  │     │ LangSmith  │
      │ LangGraph  │     │ evals,     │     │ Docker, or │     │ traces,    │
      │ Deep Agents│     │ pytest     │     │ LangSmith  │     │ dashboards │
      └────────────┘     └────────────┘     └────────────┘     └─────┬──────┘
            ▲                                                        │
            └──────── traces become new test cases and fixes ◀───────┘
```

The loop never ends: production traces show you new failures, you turn them into test cases (datasets), you fix the system, and you deploy again. This is called **evaluation-driven development**.

### Where each module sits

| Lifecycle step | Modules |
|---|---|
| Build | 01 Models & prompts · 02 Tools & agents · 03 Memory · 04 Middleware & guardrails · 05 MCP & web search · 06 RAG · 07–08 LangGraph · 09 Multi-agent · 10 Deep Agents · 11 Multimodal & voice · 14 Local models |
| Test | Lesson 01.08 (first traces) · 12 Evaluation & observability |
| Deploy | 13 Production (FastAPI + Docker; LangSmith deployment as a reference) |
| Monitor | 12 (online evaluation, dashboards) · 13 (logging, cost) |

### What just happened?

You now have the map for the whole course. Every later lesson answers one question: *which part of build → test → deploy → monitor does this improve?* If you can't answer that for a new tool you read about, you probably don't need it yet.

---
## 4. Interactive glossary

The next cells build a small glossary with plain Python. You will use it to look up terms and quiz yourself.

**Jupyter tip:** the first cell only *defines* data and functions. Nothing visible happens when you run it. That is normal: the definitions now live in the **kernel's memory** and later cells can use them.

In [ ]:
# Step 1: The glossary is a dict: term -> (plain-English definition, where the course teaches it).
# WHY a dict: lookups by key are instant, and you can add your own terms in one line.
GLOSSARY: dict[str, tuple[str, str]] = {
    "agent": ("A model that runs in a loop, deciding which tools to call until the task is done.", "Module 02"),
    "harness": ("Everything around the model loop: prompt, tools, middleware. Agent = model + harness.", "Lesson 00.01"),
    "framework": ("Abstractions and integrations for building with LLMs. LangChain is one.", "Lesson 00.01"),
    "runtime": ("Infrastructure that runs agents reliably: durable execution, streaming, persistence. LangGraph is one.", "Module 07"),
    "deep agent": ("An agent built with a batteries-included harness: planning, filesystem, subagents.", "Module 10"),
    "context engineering": ("Choosing what goes into the model's context window: instructions, data, memory, tool results.", "Module 03"),
    "context window": ("The maximum number of tokens a model can read in one request (input + output).", "Lesson 00.06"),
    "token": ("A chunk of text (often part of a word) that models read and write. Pricing and limits count tokens.", "Lesson 00.06"),
    "rag": ("Retrieval-augmented generation: fetch relevant documents first, then answer using them.", "Module 06"),
    "tool calling": ("The model asks your code to run a function with specific arguments, then reads the result.", "Module 02"),
    "middleware": ("Hooks that run before or after model and tool calls in create_agent, e.g. limits or guardrails.", "Module 04"),
    "mcp": ("Model Context Protocol: a standard way to expose tools and data to any agent.", "Module 05"),
    "observability": ("Recording what every run did (traces) so you can debug and monitor it.", "Module 12"),
    "trace": ("The recorded tree of steps (model calls, tool calls) for one run of your app.", "Lesson 01.08"),
    "evaluation-driven development": ("Change the system only when a test dataset shows the change helps.", "Module 12"),
    "human-in-the-loop": ("Pausing an agent so a person can approve, edit, or reject an action.", "Modules 04 and 08"),
    "preview model": ("A model offered for evaluation only. It can be retired at short notice. Not for production.", "Lesson 00.06"),
    "kernel": ("The process that runs a notebook's code and keeps its variables in memory.", "Lesson 00.03"),
}

print(f"Glossary loaded: {len(GLOSSARY)} terms.")

In [ ]:
import difflib


def define(term: str) -> None:
    """Print the definition of `term`, or suggest close matches if it is misspelled.

    Args:
        term: The term to look up. Case and surrounding spaces are ignored.
    """
    key = term.strip().lower()
    if key in GLOSSARY:
        definition, where = GLOSSARY[key]
        print(f"{key.upper()}\n  {definition}\n  Taught in: {where}")
        return
    # WHY difflib: typos are common; suggesting "harness" for "harnes" is friendlier than "not found".
    suggestions = difflib.get_close_matches(key, GLOSSARY.keys(), n=3, cutoff=0.6)
    hint = f" Did you mean: {', '.join(suggestions)}?" if suggestions else " Try search() instead."
    print(f"'{term}' is not in the glossary.{hint}")


def search(word: str) -> list[str]:
    """Return every term whose name or definition contains `word` (case-insensitive)."""
    word = word.lower()
    return sorted(t for t, (d, _) in GLOSSARY.items() if word in t or word in d.lower())


define("harness")
print()
define("harnes")  # Deliberate typo, to see the suggestion.
print()
print("Terms mentioning 'model':", search("model"))

### What just happened?

- `define("harness")` found the term and printed its definition.
- `define("harnes")` did not find it, so `difflib` suggested the closest term. Small touches like this are what "engineering" means.
- `search("model")` scanned every definition. Notice how many terms involve the model *and something around it*.

Now render the glossary as a nicely formatted table. `IPython.display.Markdown` turns a Markdown string into rendered output, which is how notebooks show rich results.

In [ ]:
from IPython.display import Markdown, display

rows = "\n".join(f"| **{t}** | {d} | {w} |" for t, (d, w) in sorted(GLOSSARY.items()))
display(Markdown("| Term | Definition | Taught in |\n|---|---|---|\n" + rows))

### Self-quiz

The next cell picks three random terms. Say the definition **out loud** (or write it in a Markdown cell), *then* run the cell after it to check yourself.

We pass a fixed **seed** to the random generator so the quiz is the same every time you run the notebook from the top. Change `QUIZ_SEED` to get a new quiz.

In [ ]:
import random

QUIZ_SEED = 2026  # Change me for a different quiz.
quiz_terms = random.Random(QUIZ_SEED).sample(sorted(GLOSSARY), k=3)

for number, term in enumerate(quiz_terms, start=1):
    print(f"Q{number}. What is '{term}'?")

In [ ]:
# Answers. Run this only after you have tried the quiz above.
for term in quiz_terms:
    define(term)
    print()

---
## 5. Jupyter skill: Markdown cells

Every text cell in this notebook is a **Markdown cell**. Course notebooks use them for explanations, and you will use them for notes and project findings.

**Try it now:**

1. Click once on the left margin of this cell so it is selected (blue bar on the left). You are in **command mode**.
2. Press **Enter** (or double-click the text) to enter **edit mode**. You now see the raw Markdown.
3. Press **Shift + Enter** to render it again.
4. To create your own: press **Esc**, then **B** (new cell *below*), then **M** (make it Markdown). Type, then **Shift + Enter**.

| You type | You get |
|---|---|
| `# Title`, `## Section` | Headings (the notebook's table of contents uses them) |
| `**bold**`, `*italic*`, `` `code` `` | **bold**, *italic*, `code` |
| `- item` or `1. item` | Bullet or numbered lists |
| `[text](https://example.com)` | A link |
| `> quote` | A callout, like the ⚠️ Production notes |
| `$E = mc^2$` | Inline math with LaTeX: $E = mc^2$ |
| `$$\sum_{i=1}^{n} t_i$$` | Display math: $$\text{cost} = \sum_{i=1}^{n} \text{tokens}_i \times \text{price}$$ |

Markdown reference used by Jupyter: [Markdown cells (Jupyter Notebook docs)](https://jupyter-notebook.readthedocs.io/en/stable/notebook.html).

---
## ⚠️ Pitfalls and misconceptions

| Misconception | Reality |
|---|---|
| "LangChain is outdated; everyone uses LangGraph now." | They are layers, not rivals. LangChain's `create_agent` runs on LangGraph. You will use both. |
| "A better model fixes everything." | Most failures come from missing context, bad tools, or no evaluation, not from the model. |
| "If the notebook works once, it works." | Only **Restart & Run All** proves it (Lesson 00.03). |
| "Tutorials from last year are fine." | Model IDs and APIs change monthly. Lesson 00.05 teaches you to verify. |

---
## 🏋️ Exercises

1. **(Easy)** Add two terms of your own to `GLOSSARY` in a **new code cell** below the glossary cells, for example `"llm"` and `"prompt"`. Run `define()` on them.
2. **(Easy)** Create a Markdown cell with a three-row table: *Runtime / Framework / Harness* and one tool for each.
3. **(Medium)** Write a function `quiz(n)` that prints `n` random questions using `random.Random(QUIZ_SEED)`. What happens if `n` is larger than the glossary? Make the function handle that.
4. **(Stretch)** Pick one row of the "What breaks" table and write 3–4 sentences in a Markdown cell: how would you *detect* that failure in production?

<details>
<summary><b>Show solutions</b> (try first!)</summary>

**1.**
```python
GLOSSARY["llm"] = ("Large language model: predicts the next token of text.", "Module 01")
GLOSSARY["prompt"] = ("The instructions and data you send to a model.", "Module 01")
define("llm")
```

**3.**
```python
def quiz(n: int) -> None:
    n = min(n, len(GLOSSARY))  # sample() raises ValueError if k > population size
    for i, term in enumerate(random.Random(QUIZ_SEED).sample(sorted(GLOSSARY), k=n), 1):
        print(f"Q{i}. What is '{term}'?")
```

**4.** Example for "model ID disappears": run a daily job that lists live models (Project 00) and alerts when a configured ID is missing; also alert on `model_not_found` / HTTP 404 errors in logs.
</details>

---
## ✅ Checkpoint

You are ready for the next lesson if you can:

- [ ] Explain "Agent = model + harness" in one sentence.
- [ ] Name the four lifecycle steps and one course module for each.
- [ ] Say which of LangChain / LangGraph / Deep Agents is the runtime, the framework, and the harness.
- [ ] Create, edit, and render a Markdown cell.

## 🔎 Research checkpoint (5 minutes)

Open [Runtimes, frameworks, and harnesses](https://docs.langchain.com/oss/python/concepts/products). The page lists *other* examples of frameworks, runtimes, and harnesses besides LangChain's. Write down one of each, and note it in [`docs/research-log.md`](../../docs/research-log.md) (the module-level entry comes at the end of Lesson 00.07).

## Summary

- An AI engineer builds the **system around the model**: context, tools, retrieval, evaluation, observability, deployment.
- **Agent = model + harness.** LangGraph is the runtime, LangChain the framework, Deep Agents the harness. LangSmith traces, evaluates, and deploys all of them.
- The work is a loop: **build → test → deploy → monitor**, and production traces feed the next round of tests.
- Markdown cells are how you explain your work. Use them generously.

**Next:** [00.02 · Verify your environment](00-02-verify-environment.ipynb). Before you open it, keep this notebook's kernel in mind: the next lesson is about *which Python* actually runs your code.